In [ ]:
import os
from sklearn.pipeline import Pipeline
from model_class.feature_builder import FeatureBuilder
from model_class.feature_builder_transformer import FeatureBuilderTransformer
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score, roc_curve


### 1. Import data

In [ ]:
# Get the current directory
current_directory = os.getcwd()

# Get the source file path
file_path = f'{current_directory}/datasets/raw'
file = f"{file_path}/labelled_training_data.csv"
df_train = pd.read_csv(file, header=0)
y_train = df_train['evil']
X_train = df_train.drop(columns=['sus', 'evil'])

# Get the source file path
file_path = f'{current_directory}/datasets/raw'
file = f"{file_path}/labelled_testing_data.csv"
df_test = pd.read_csv(file, header=0)
y_test = df_test['evil']
X_test = df_test.drop(columns=['sus', 'evil'])

# Get the source file path
file_path = f'{current_directory}/datasets/raw'
file = f"{file_path}/labelled_validation_data.csv"
df_validation = pd.read_csv(file, header=0)
y_validation = df_validation['evil']
X_validation = df_validation.drop(columns=['sus', 'evil'])

### 2. build features

In [ ]:
scale_cols = [
    "processId_freq",
    "threadId_freq",
    "parentProcessId_freq",
    "userId_freq",
    "mountNamespace_freq",
    "eventId_freq",
    "stackAddresses_len",
    "returnValue",
    "argsNum",
    "stackAddresses_jump_std"
]

passthrough_cols = [
    # "hostName_hash",
    # "processName_hash",
    # "parentProcessName_hash",
    "is_system_process",
    "is_parent_system_process",
    "userId_binary",
    "parentUserId_binary",
    "same_user_as_parent",
    "same_process_name_as_parent",
    "stackAddresses_unique_ratio",
    "returnValue_is_error",
    "args_has_path",
    "mountNamespace_binary"
]



scaler = ColumnTransformer(
    transformers=[
        ("scale", RobustScaler(), scale_cols),
        ("pass", "passthrough", passthrough_cols),
    ]
)


1. Isolation Forest

In [ ]:
model = Pipeline([
    ("features", FeatureBuilderTransformer(FeatureBuilder(), return_numpy=False)),
    ("scaler", scaler),
    ("iforest", IsolationForest(
            n_estimators=100,
            contamination=0.001,
            max_features=0.7,
            random_state=2000
        ))

])

In [ ]:
# Perform fit on X _train and returns labels for y_pred.
y_pred_train = model.fit_predict(X_train)

# Perform fit on X _train and returns labels for y_pred.
y_pred_validation= model.predict(X_validation)

# Perform fit on X _train and returns labels for y_pred.
y_pred_test = model.predict(X_test)

### Model Evaluation

In [ ]:

def evaluate_model(y_true, y_pred, dataset_type="Dataset"):
    # IsolationForest: -1 => outlier, 1 => inlier
    # Map to binary where 1 means anomaly/outlier and 0 means inlier/normal
    y_pred_mapped = np.where(y_pred == -1, 1, 0)

    # Ensure y_true is numeric binary array
    y_true_arr = np.array(y_true).astype(int)

    # Basic counts
    unique_values, counts = np.unique(y_true_arr, return_counts=True)
    print(f"y_true_{dataset_type} Unique values: {unique_values}")
    print(f"y_true_{dataset_type} Counts of each value: {counts}")

    unique_values, counts = np.unique(y_pred_mapped, return_counts=True)
    print(f"y_pred_{dataset_type} Unique values: {unique_values}")
    print(f"y_pred_{dataset_type} Counts of each value: {counts}")

    # Confusion Matrix (labels assumed [0,1] -> [[TN, FP], [FN, TP]])
    cm = confusion_matrix(y_true_arr, y_pred_mapped)
    print(f"Confusion matrix for {dataset_type}:\n{cm}")
    try:
        tn, fp, fn, tp = cm.ravel()
    except Exception:
        tn = fp = fn = tp = None

    # Simple accuracy
    total = len(y_true_arr)
    correct = int((y_true_arr == y_pred_mapped).sum())
    accuracy = correct / total * 100 if total > 0 else 0.0
    print(f"Accuracy for {dataset_type}: {correct}/{total} = {accuracy:.2f}%")

    # Print detailed counts if available
    if tn is not None:
        print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")

    return accuracy

In [ ]:
evaluate_model(y_train, y_pred_train, dataset_type="Training Dataset")
evaluate_model(y_validation, y_pred_validation, dataset_type="Validation Dataset")
evaluate_model(y_test, y_pred_test, dataset_type="Test Dataset")